# Databricks Local Development Example

This notebook demonstrates how to run PySpark code locally using Databricks local mode.

In [1]:
# Import required libraries
import sys
import os

# Add src to path for imports
sys.path.append('../src')

from utils import get_spark_session, setup_local_environment

In [2]:
from pyspark.sql.functions import col, count, avg, max as spark_max

In [3]:
# Setup local environment
setup_local_environment()

# Create Spark session
spark = get_spark_session("Local Development Example")
print(f"Spark version: {spark.version}")
print(f"Spark context: {spark.sparkContext.appName}")

🔧 Local environment configured for Databricks development


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/30 17:27:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0
Spark context: Local Development Example


In [4]:
# Create sample data
sample_data = [
    ("Product A", "Electronics", 299.99, 150),
    ("Product B", "Clothing", 49.99, 200),
    ("Product C", "Electronics", 799.99, 75),
    ("Product D", "Books", 19.99, 300),
    ("Product E", "Clothing", 89.99, 120),
    ("Product F", "Electronics", 1299.99, 25),
    ("Product G", "Books", 29.99, 180),
]

columns = ["product_name", "category", "price", "quantity_sold"]
df = spark.createDataFrame(sample_data, columns)

print("Sample dataset:")
df.show()

Sample dataset:


+------------+-----------+-------+-------------+
|product_name|   category|  price|quantity_sold|
+------------+-----------+-------+-------------+
|   Product A|Electronics| 299.99|          150|
|   Product B|   Clothing|  49.99|          200|
|   Product C|Electronics| 799.99|           75|
|   Product D|      Books|  19.99|          300|
|   Product E|   Clothing|  89.99|          120|
|   Product F|Electronics|1299.99|           25|
|   Product G|      Books|  29.99|          180|
+------------+-----------+-------+-------------+



In [5]:
# Basic data exploration
print(f"Total rows: {df.count()}")
print(f"Columns: {df.columns}")
print("\nSchema:")
df.printSchema()

Total rows: 7
Columns: ['product_name', 'category', 'price', 'quantity_sold']

Schema:
root
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity_sold: long (nullable = true)



In [6]:
# Data transformations
# Add revenue column
df_with_revenue = df.withColumn("revenue", col("price") * col("quantity_sold"))

print("Data with revenue column:")
df_with_revenue.show()

Data with revenue column:
+------------+-----------+-------+-------------+-----------------+
|product_name|   category|  price|quantity_sold|          revenue|
+------------+-----------+-------+-------------+-----------------+
|   Product A|Electronics| 299.99|          150|          44998.5|
|   Product B|   Clothing|  49.99|          200|           9998.0|
|   Product C|Electronics| 799.99|           75|         59999.25|
|   Product D|      Books|  19.99|          300|5996.999999999999|
|   Product E|   Clothing|  89.99|          120|          10798.8|
|   Product F|Electronics|1299.99|           25|         32499.75|
|   Product G|      Books|  29.99|          180|           5398.2|
+------------+-----------+-------+-------------+-----------------+



In [7]:
# Aggregations by category
category_summary = (
    df_with_revenue
    .groupBy("category")
    .agg(
        count("*").alias("product_count"),
        avg("price").alias("avg_price"),
        spark_max("price").alias("max_price"),
        avg("quantity_sold").alias("avg_quantity_sold"),
        avg("revenue").alias("avg_revenue")
    )
    .orderBy(col("avg_revenue").desc())
)

print("Category summary:")
category_summary.show(truncate=False)

Category summary:
+-----------+-------------+-----------------+---------+-----------------+-----------------+
|category   |product_count|avg_price        |max_price|avg_quantity_sold|avg_revenue      |
+-----------+-------------+-----------------+---------+-----------------+-----------------+
|Electronics|3            |799.9900000000001|1299.99  |83.33333333333333|45832.5          |
|Clothing   |2            |69.99            |89.99    |160.0            |10398.4          |
|Books      |2            |24.99            |29.99    |240.0            |5697.599999999999|
+-----------+-------------+-----------------+---------+-----------------+-----------------+

+-----------+-------------+-----------------+---------+-----------------+-----------------+
|category   |product_count|avg_price        |max_price|avg_quantity_sold|avg_revenue      |
+-----------+-------------+-----------------+---------+-----------------+-----------------+
|Electronics|3            |799.9900000000001|1299.99  |83.333

In [8]:
# Filter high-value products
high_value_products = df_with_revenue.filter(col("price") > 100)

print("High-value products (>$100):")
high_value_products.select("product_name", "category", "price", "revenue").show()

High-value products (>$100):
+------------+-----------+-------+--------+
|product_name|   category|  price| revenue|
+------------+-----------+-------+--------+
|   Product A|Electronics| 299.99| 44998.5|
|   Product C|Electronics| 799.99|59999.25|
|   Product F|Electronics|1299.99|32499.75|
+------------+-----------+-------+--------+



In [9]:
# SQL queries
df_with_revenue.createOrReplaceTempView("products")

sql_result = spark.sql("""
    SELECT 
        category,
        COUNT(*) as product_count,
        SUM(revenue) as total_revenue,
        AVG(price) as avg_price
    FROM products
    GROUP BY category
    ORDER BY total_revenue DESC
""")

print("SQL query results:")
sql_result.show()

SQL query results:
+-----------+-------------+------------------+-----------------+
|   category|product_count|     total_revenue|        avg_price|
+-----------+-------------+------------------+-----------------+
|Electronics|            3|          137497.5|799.9900000000001|
|   Clothing|            2|           20796.8|            69.99|
|      Books|            2|11395.199999999999|            24.99|
+-----------+-------------+------------------+-----------------+



In [10]:
# Save results (optional)
# output_path = "../data/category_summary"
# category_summary.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)
# print(f"Results saved to {output_path}")

print("✅ Notebook execution completed successfully!")

✅ Notebook execution completed successfully!


In [11]:
# Clean up
spark.stop()
print("Spark session stopped.")

Spark session stopped.
